In [ ]:
import cv2
from ultralytics import YOLO
import socket
import json
from signal_processor import  SignalProcessor  

# Socket 설정
sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
server_address = ('localhost', 9999)

# 로우패스 필터
signal_processor1 = SignalProcessor(window_size=5, alpha=.2)
signal_processor2 = SignalProcessor(window_size=5, alpha=.2)

model = YOLO("yolo26m-seg.pt")
#source = "cat.mp4"  # 웹캠은 0, 동영상 파일은 'video.mp4'
#source = 'http://10.10.14.50:5000/video_feed'
source = 'http://10.10.14.11:5000/video_feed'


cap = cv2.VideoCapture(source)

width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
cam_x = int(width // 2)
cam_y = int(height // 2)

print("start")

try:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            print("no frame")
            pass

        #frame = cv2.flip(frame, 0) #카메라 뒤집기

        # 추론 41컵
        results = model.predict(frame, classes=[41], stream=True, verbose=False, device=0,  conf=0.4)

        for result in results:   
            annotated_frame = result.plot()
            error = 0

            for i, box in enumerate(result.boxes):
                class_id = int(box.cls[0]) # 클래스 id 흭득
           
                # 컵
                if class_id == 41:
                    # 해당 객체의 마스크 좌표
                    xy = result.masks.xy[i]

                    if len(xy) > 0:
                        cx = int(xy[:, 0].mean()) # x를 모두 구한거의 평균
                        cy = int(xy[:, 1].mean()) # y를 모두 구한거의 평균

                        # low pass 필터 적용
                        lowpass_cx = int(signal_processor1.moving_average(cx))
                        lowpass_cy = int(signal_processor2.moving_average(cy))

                        # 오차 계산
                        error = cam_x - lowpass_cx

                        # 디버깅용
                        cv2.circle(annotated_frame, (lowpass_cx, lowpass_cy), 5, (0, 0, 255), -1)
                        cv2.putText(annotated_frame, f"x axis error: {error} ", (lowpass_cx, lowpass_cy - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

                        # 데이터를 JSON으로 직렬화해서 socket 전송
                        data = json.dumps(error).encode('utf-8')
                        sock.sendto(data, server_address)
                        print(f"Sent: {(error)}")

        cv2.line(annotated_frame, (cam_x, 0), (cam_x, cam_y*2), (0, 0, 255), 1)
        cv2.imshow("YOLO ", annotated_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
        
except KeyboardInterrupt:
    print("END.")

finally:
    cap.release()
    cv2.destroyAllWindows() 